In [1]:
import pandas as pd
from pathlib import Path

file_path = Path("../data/raw/phishing_dataset.csv")

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (6000, 8)

Column names:
['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label', 'source']

First 5 rows:


,sender,receiver,date,subject,body,urls,label,source
0,mrssuha1122@yahoo.co.uk,R@M,"Sat, 03 Mar 2007 07:58:25 -0800",Urgent,This item from National Association of the Dea...,1.0,1,nigerian_fraud
1,"""Michael Moncur"" <mgm@starlingtech.com>","""Justin Mason"" <yyyy@spamassassin.taint.org>,","Thu, 19 Sep 2002 21:42:52 -0600",RE: [SAdev] phew!,"> Yes, I think some others, (mgm?) have spamtr...",1.0,0,spamassassin
2,Apple <apple-tikung@appleid-apple-supports.ipb...,jose@monkey.org,24 Sep 2017 13:10:51 +0000,Reminder : Apple ID has been locked.,Dear jose@monkey.orgWe've detect that your App...,1.0,1,nazario
3,NaN,ejimegoalota@yahoo.com,NaN,URGENT ASSISTANCE,"\nFROM THE DESK OF BILL AND EXCHANGE MANAGER,\...",0.0,1,nigerian_fraud
4,pudge@perl.org,yyyy-use-perl@spamassassin.taint.org,"Wed, 28 Aug 2002 02:00:23 +0000",[use Perl] Headlines for 2002-08-28,use Perl Daily Headline Mailer\n\n.NET and Per...,1.0,0,spamassassin



Missing values:
sender      162
receiver    790
date        222
subject      27
body          0
urls          0
label         0
source        0
dtype: int64


In [2]:
# 1. Check the label categories and counts
print("Label distribution:")
print(df["label"].value_counts())

# 2. Check the data types
print("\nData types:")
print(df.dtypes)

# 3. Inspect sample emails and their labels
display(df[["sender", "subject", "body", "urls", "label"]].head(10))

# 4. Check whether URLs contains actual URLs or just counts
print("\nSample URL values:")
print(df["urls"].head(10).tolist())

# 5. Check duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

Label distribution:
label
1    3000
0    3000
Name: count, dtype: int64

Data types:
sender          str
receiver        str
date            str
subject         str
body            str
urls        float64
label         int64
source          str
dtype: object


,sender,subject,body,urls,label
0,mrssuha1122@yahoo.co.uk,Urgent,This item from National Association of the Dea...,1.0,1
1,"""Michael Moncur"" <mgm@starlingtech.com>",RE: [SAdev] phew!,"> Yes, I think some others, (mgm?) have spamtr...",1.0,0
2,Apple <apple-tikung@appleid-apple-supports.ipb...,Reminder : Apple ID has been locked.,Dear jose@monkey.orgWe've detect that your App...,1.0,1
3,NaN,URGENT ASSISTANCE,"\nFROM THE DESK OF BILL AND EXCHANGE MANAGER,\...",0.0,1
4,pudge@perl.org,[use Perl] Headlines for 2002-08-28,use Perl Daily Headline Mailer\n\n.NET and Per...,1.0,0
5,miss peace <peace_mathew10@yahoo.com>,"PLEASE STAND ON MY BEHALF,","Dear Friend, Kindly accept my apology for send...",1.0,1
6,monkey.org support<sales@k-fuss.life>,"Alert: About your account suspension! Monday, ...",monkey.org update! This is to notify your acco...,1.0,1
7,mustapher mustapher <mustapher_30@hotmail.com>,please get back to me.,"ATTN.PLEASE\n\n\nI am mustapher ahmed, I wor...",1.0,1
8,"""CapitalOne 360"" <membershiprelationship@capit...",Important Security Alert!,"Hello, We just wanted to let you know that you...",1.0,1
9,monkey.org<miac-lax@mail-wto.com>,Email-Quota Limit Report,HTML Editor - Full Version monkey.org Webmail ...,1.0,1



Sample URL values:
[1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

Duplicate rows: 2


In [3]:
# Check label distribution by source
print(pd.crosstab(df["source"], df["label"]))

# Compare average URL counts by label
print(df.groupby("label")["urls"].describe())

# Inspect examples from each label
for label in sorted(df["label"].unique()):
    print(f"\n--- Label {label} ---")
    display(
        df.loc[df["label"] == label,
               ["sender", "subject", "body", "urls", "source"]].head(3)
    )

label              0     1
source                    
nazario            0   738
nigerian_fraud     0  1483
spamassassin    3000   779
        count      mean       std  min  25%  50%  75%  max
label                                                     
0      3000.0  0.884000  0.320278  0.0  1.0  1.0  1.0  1.0
1      3000.0  0.603667  0.489217  0.0  0.0  1.0  1.0  1.0

--- Label 0 ---


,sender,subject,body,urls,source
1,"""Michael Moncur"" <mgm@starlingtech.com>",RE: [SAdev] phew!,"> Yes, I think some others, (mgm?) have spamtr...",1.0,spamassassin
4,pudge@perl.org,[use Perl] Headlines for 2002-08-28,use Perl Daily Headline Mailer\n\n.NET and Per...,1.0,spamassassin
10,Rick Baartman <baartman@lin12.triumf.ca>,Re: Sorting,Thanks Brent: now it's clearer (to me) what's ...,1.0,spamassassin



--- Label 1 ---


,sender,subject,body,urls,source
0,mrssuha1122@yahoo.co.uk,Urgent,This item from National Association of the Dea...,1.0,nigerian_fraud
2,Apple <apple-tikung@appleid-apple-supports.ipb...,Reminder : Apple ID has been locked.,Dear jose@monkey.orgWe've detect that your App...,1.0,nazario
3,NaN,URGENT ASSISTANCE,"\nFROM THE DESK OF BILL AND EXCHANGE MANAGER,\...",0.0,nigerian_fraud


In [4]:
# Make a working copy
clean_df = df.copy()

# Remove exact duplicate rows
clean_df = clean_df.drop_duplicates().copy()

# Fill missing text fields
text_columns = ["sender", "receiver", "date", "subject", "body", "source"]

for col in text_columns:
    clean_df[col] = clean_df[col].fillna("")

# Ensure URL count is numeric
clean_df["urls"] = pd.to_numeric(
    clean_df["urls"], errors="coerce"
).fillna(0)

# Check the result
print("Original rows:", len(df))
print("Rows after removing duplicates:", len(clean_df))
print("\nMissing values:")
print(clean_df.isnull().sum())

print("\nLabel distribution:")
print(clean_df["label"].value_counts())

Original rows: 6000
Rows after removing duplicates: 5998

Missing values:
sender      0
receiver    0
date        0
subject     0
body        0
urls        0
label       0
source      0
dtype: int64

Label distribution:
label
0    3000
1    2998
Name: count, dtype: int64


In [5]:
# Inspect sender and domain patterns
print("Sample sender addresses:")
print(clean_df["sender"].head(10).to_list())

# Inspect URLs and their counts
print("\nURL count statistics:")
print(clean_df["urls"].describe())

# Check the most common sender domains
clean_df["sender_domain"] = (
    clean_df["sender"]
    .str.extract(r'@([^>\s]+)')[0]
    .str.lower()
)

print("\nMost common sender domains:")
print(clean_df["sender_domain"].value_counts().head(10))

# Compare labels by source
print("\nLabel distribution by source:")
print(pd.crosstab(clean_df["source"], clean_df["label"]))

Sample sender addresses:
['mrssuha1122@yahoo.co.uk', '"Michael Moncur" <mgm@starlingtech.com>', 'Apple <apple-tikung@appleid-apple-supports.ipbvs.com>', '', 'pudge@perl.org', 'miss peace <peace_mathew10@yahoo.com>', 'monkey.org support<sales@k-fuss.life>', 'mustapher mustapher <mustapher_30@hotmail.com>', '"CapitalOne 360" <membershiprelationship@capitalone360.com>', 'monkey.org<miac-lax@mail-wto.com>']

URL count statistics:
count    5998.000000
mean        0.744081
std         0.436413
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: urls, dtype: float64

Most common sender domains:
sender_domain
spamassassin.taint.org    477
hotmail.com               285
yahoo.com                 196
yahoo.co.uk                72
virgilio.it                71
slack.net                  60
canada.com                 58
newsletter.online.com      57
perl.org                   56
netscape.net               55
Name: count, dtype: int64

Label

In [6]:
import re
from urllib.parse import urlparse

# Start with a copy of the cleaned dataset
feature_df = clean_df.copy()

# Combine subject and body for text analysis
feature_df["email_text"] = (
    feature_df["subject"] + " " + feature_df["body"]
)

# Extract sender domain
feature_df["sender_domain"] = (
    feature_df["sender"]
    .str.extract(r'@([^>\s]+)')[0]
    .fillna("")
    .str.lower()
)

# Count URLs found in the email body
feature_df["extracted_url_count"] = (
    feature_df["body"]
    .str.count(r'https?://|www\.')
)

# Detect urgent language
urgent_words = r'\b(urgent|immediately|suspended|verify|password|account)\b'

feature_df["urgent_word_count"] = (
    feature_df["email_text"]
    .str.lower()
    .str.count(urgent_words)
)

# Email text length
feature_df["email_length"] = (
    feature_df["email_text"].str.len()
)

# Check the extracted features
display(feature_df[
    ["sender_domain", "urls", "extracted_url_count",
     "urgent_word_count", "email_length", "label"]
].head(10))

,sender_domain,urls,extracted_url_count,urgent_word_count,email_length,label
0,yahoo.co.uk,1.0,2,3,1507,1
1,starlingtech.com,1.0,2,0,447,0
2,appleid-apple-supports.ipbvs.com,1.0,0,0,562,1
3,,0.0,0,2,1519,1
4,perl.org,1.0,2,0,642,0
5,yahoo.com,1.0,2,1,1527,1
6,k-fuss.life,1.0,0,3,451,1
7,hotmail.com,1.0,0,1,1523,1
8,capitalone360.com,1.0,0,11,1526,1
9,mail-wto.com,1.0,0,0,368,1


In [7]:
print("Dataset source information:")
print(clean_df["source"].value_counts())

print("\nLabel counts by source:")
print(pd.crosstab(clean_df["source"], clean_df["label"]))

print("\nCheck dataset documentation or README to confirm:")
print("Does label 1 mean phishing and label 0 mean legitimate?")

Dataset source information:
source
spamassassin      3779
nigerian_fraud    1481
nazario            738
Name: count, dtype: int64

Label counts by source:
label              0     1
source                    
nazario            0   738
nigerian_fraud     0  1481
spamassassin    3000   779

Check dataset documentation or README to confirm:
Does label 1 mean phishing and label 0 mean legitimate?


In [8]:
# Create the feature dataset
model_df = feature_df.copy()

# Keep only the columns needed for initial model training
model_df = model_df[
    [
        "email_text",
        "sender_domain",
        "urls",
        "urgent_word_count",
        "email_length",
        "label"
    ]
].copy()

# Confirm there are no missing values
print("Missing values:")
print(model_df.isnull().sum())

# Check class balance
print("\nClass distribution:")
print(model_df["label"].value_counts())

# Save the prepared dataset
model_df.to_csv(
    "../data/processed/prepared_emails.csv",
    index=False
)

print("\nSaved to data/processed/prepared_emails.csv")
print("Final shape:", model_df.shape)

Missing values:
email_text           0
sender_domain        0
urls                 0
urgent_word_count    0
email_length         0
label                0
dtype: int64

Class distribution:
label
0    3000
1    2998
Name: count, dtype: int64

Saved to data/processed/prepared_emails.csv
Final shape: (5998, 6)


In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Load prepared data
df_model = pd.read_csv("../data/processed/prepared_emails.csv")

# Separate features and target
X = df_model[
    [
        "email_text",
        "sender_domain",
        "urls",
        "urgent_word_count",
        "email_length"
    ]
]
y = df_model["label"]

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Process text and other features separately
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=5000), "email_text"),
        ("sender", OneHotEncoder(handle_unknown="ignore"), ["sender_domain"]),
        ("numeric", StandardScaler(), [
            "urls",
            "urgent_word_count",
            "email_length"
        ])
    ]
)

# Build the model pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       600
           1       0.99      0.98      0.98       600

    accuracy                           0.98      1200
   macro avg       0.99      0.98      0.98      1200
weighted avg       0.99      0.98      0.98      1200

Confusion Matrix:
[[592   8]
 [ 10 590]]


In [10]:
import joblib
from pathlib import Path

model_path = Path("../models/phishing_model.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(model, model_path)

print("Model saved to:", model_path.resolve())

Model saved to: C:\Users\PRANJALI\Desktop\AI Phishing Detection\models\phishing_model.joblib


In [11]:
import pandas as pd

test_email = pd.DataFrame([{
    "email_text": (
        "URGENT: Your account has been suspended. "
        "Click here to verify your password immediately."
    ),
    "sender_domain": "account-security-example.com",
    "urls": 1,
    "urgent_word_count": 3,
    "email_length": 100
}])

prediction = model.predict(test_email)[0]
probabilities = model.predict_proba(test_email)[0]

print("Prediction:", "Phishing/Spam" if prediction == 1 else "Legitimate")
print("Class probabilities:", probabilities)
print("Phishing/Spam score:", round(probabilities[1] * 100, 2), "%")

Prediction: Phishing/Spam
Class probabilities: [0.01714343 0.98285657]
Phishing/Spam score: 98.29 %


In [16]:
# Rebuild aligned data
evaluation_df = model_df.copy()
evaluation_df["source"] = clean_df["source"].values

# Split row indices
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    evaluation_df.index,
    test_size=0.2,
    random_state=42,
    stratify=evaluation_df["label"]
)

# Evaluate each source using the existing model's test predictions
test_results = evaluation_df.loc[test_idx].copy()
test_results["predicted"] = model.predict(
    test_results[X.columns]
)

for source, group in test_results.groupby("source"):
    print(f"\nSource: {source}")
    print(classification_report(
        group["label"],
        group["predicted"],
        zero_division=0
    ))


Source: nazario
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.99      0.99       139

    accuracy                           0.99       139
   macro avg       0.50      0.49      0.50       139
weighted avg       1.00      0.99      0.99       139


Source: nigerian_fraud
              precision    recall  f1-score   support

           1       1.00      1.00      1.00       307

    accuracy                           1.00       307
   macro avg       1.00      1.00      1.00       307
weighted avg       1.00      1.00      1.00       307


Source: spamassassin
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       600
           1       0.95      0.95      0.95       154

    accuracy                           0.98       754
   macro avg       0.97      0.97      0.97       754
weighted avg       0.98      0.98      0.98       754



In [17]:
import joblib
from pathlib import Path

model_path = Path("../models/phishing_model.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(model, model_path)

print("Model saved successfully!")
print("Location:", model_path.resolve())

Model saved successfully!
Location: C:\Users\PRANJALI\Desktop\AI Phishing Detection\models\phishing_model.joblib


In [2]:
import joblib
from pathlib import Path

model_path = Path("../models/phishing_model.joblib")

model = joblib.load(model_path)

print("Model loaded successfully!")

Model loaded successfully!


In [3]:
# Test the legitimate email again
import pandas as pd

test_email = pd.DataFrame([{
    "email_text": """Subject: Reminder: Upcoming Semester Project Submission

Dear Students,

This is a reminder that the semester project submission deadline is approaching.

Please ensure that your project documentation and presentation are ready before Friday.

You can contact your project coordinator if you have any questions regarding the submission process.

We wish you all the best.

Regards,
Department of Computer Science""",

    "sender_domain": "university.edu",
    "urls": 0,
    "urgent_word_count": 0,
    "email_length": 400
}])

prediction = model.predict(test_email)[0]
probability = model.predict_proba(test_email)[0]

print("Prediction:", prediction)
print("Legitimate probability:", probability[0])
print("Phishing probability:", probability[1])

Prediction: 1
Legitimate probability: 0.15912608390182126
Phishing probability: 0.8408739160981787


In [5]:
import pandas as pd
import joblib
from pathlib import Path

# Load prepared dataset
model_df = pd.read_csv("../data/processed/prepared_emails.csv")

# Load trained model
model = joblib.load("../models/phishing_model.joblib")

print("Dataset and model loaded successfully!")
print("Dataset shape:", model_df.shape)

Dataset and model loaded successfully!
Dataset shape: (5998, 6)


In [6]:
# Get 5 legitimate and 5 phishing examples
samples = pd.concat([
    model_df[model_df["label"] == 0].sample(5, random_state=10),
    model_df[model_df["label"] == 1].sample(5, random_state=20)
])

# Predict using the trained model
X_samples = samples.drop(columns=["label"])

predictions = model.predict(X_samples)
probabilities = model.predict_proba(X_samples)[:, 1]

# Display results
results = pd.DataFrame({
    "Actual": samples["label"].values,
    "Predicted": predictions,
    "Phishing_Score": (probabilities * 100).round(2)
})

display(results)

,Actual,Predicted,Phishing_Score
0,0,0,4.84
1,0,0,9.44
2,0,0,3.21
3,0,0,20.63
4,0,0,1.83
5,1,1,99.37
6,1,1,95.30
7,1,1,98.95
8,1,1,99.91
9,1,0,47.26


In [7]:
# Get trained TF-IDF and Logistic Regression components

vectorizer = model.named_steps["preprocessor"].named_transformers_["text"]

classifier = model.named_steps["classifier"]

feature_names = vectorizer.get_feature_names_out()

# Get coefficients for phishing class
text_coefficients = classifier.coef_[0][
    :len(feature_names)
]

# Top phishing-associated words
top_indices = text_coefficients.argsort()[-20:][::-1]

print("Top words associated with phishing:\n")

for i in top_indices:
    print(feature_names[i], round(text_coefficients[i], 3))

Top words associated with phishing:

your 4.772
you 3.386
our 2.7
we 2.297
click 2.277
sightings 2.228
monkey 2.142
jose 1.949
please 1.923
my 1.799
money 1.78
will 1.593
here 1.514
email 1.495
free 1.476
bank 1.432
de 1.388
business 1.377
below 1.376
dear 1.343


In [8]:
# Top words associated with legitimate emails

bottom_indices = text_coefficients.argsort()[:20]

print("Top words associated with legitimate emails:\n")

for i in bottom_indices:
    print(feature_names[i], round(text_coefficients[i], 3))

Top words associated with legitimate emails:

re -3.085
2002 -2.718
it -2.628
the -2.605
linux -2.323
wrote -2.132
rpm -1.956
that -1.929
razor -1.895
list -1.835
users -1.781
but -1.728
they -1.649
spam -1.601
what -1.508
ie -1.472
there -1.45
mailman -1.418
exmh -1.409
fork -1.383


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X = model_df[
    ["email_text", "sender_domain", "urls",
     "urgent_word_count", "email_length"]
]

y = model_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

preprocessor = ColumnTransformer([
    ("text", TfidfVectorizer(
        max_features=10000,
        stop_words="english",
        ngram_range=(1, 2),
        sublinear_tf=True
    ), "email_text"),

    ("sender", OneHotEncoder(handle_unknown="ignore"),
     ["sender_domain"]),

    ("numeric", StandardScaler(),
     ["urls", "urgent_word_count", "email_length"])
])

improved_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

improved_model.fit(X_train, y_train)

y_pred = improved_model.predict(X_test)

print("Improved Model Results:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Improved Model Results:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       600
           1       0.99      0.99      0.99       600

    accuracy                           0.99      1200
   macro avg       0.99      0.99      0.99      1200
weighted avg       0.99      0.99      0.99      1200

Confusion Matrix:
[[593   7]
 [  8 592]]


In [10]:
import joblib
from pathlib import Path

model_path = Path("../models/improved_phishing_model.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(improved_model, model_path)

print("Improved model saved successfully!")
print(model_path.resolve())

Improved model saved successfully!
C:\Users\PRANJALI\Desktop\AI Phishing Detection\models\improved_phishing_model.joblib


In [11]:
import joblib
import pandas as pd

# Load both models
old_model = joblib.load("../models/phishing_model.joblib")
new_model = joblib.load("../models/improved_phishing_model.joblib")

# Test emails
test_emails = pd.DataFrame([
    {
        "email_text": """
        Dear Student,

        Your university examination timetable is now available.
        Please visit the official student portal to view the schedule.

        Regards,
        Examination Department
        """,
        "sender_domain": "university.edu",
        "urls": 1,
        "urgent_word_count": 0,
        "email_length": 200
    },
    {
        "email_text": """
        URGENT! Your bank account will be suspended today.
        Click here immediately to verify your account details.
        Failure to respond will result in permanent account closure.

        Verify now!
        """,
        "sender_domain": "secure-bank-alert.xyz",
        "urls": 1,
        "urgent_word_count": 4,
        "email_length": 200
    }
])

# Compare predictions
for name, model in [
    ("Old Model", old_model),
    ("Improved Model", new_model)
]:
    probabilities = model.predict_proba(test_emails)[:, 1]

    print(f"\n{name}")
    for i, score in enumerate(probabilities):
        print(
            f"Email {i+1}: "
            f"{'Phishing/Spam' if score >= 0.5 else 'Legitimate'} "
            f"({score*100:.2f}%)"
        )


Old Model
Email 1: Phishing/Spam (67.96%)
Email 2: Phishing/Spam (99.27%)

Improved Model
Email 1: Phishing/Spam (71.16%)
Email 2: Phishing/Spam (98.80%)
